# Exploration LLMs Chatbot
Thử nghiệm và khám phá dữ liệu character LLM.

In [5]:
import sys
import os

# Chỉ cần nhảy ngược ra 1 cấp để từ 'notebooks' ra 'character-llm'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..')) 

if root_path not in sys.path:
    sys.path.append(root_path)

print("Thư mục cha hiện tại:", root_path)
print("Danh sách file/folder tại đây:", os.listdir(root_path))

# Bây giờ import sẽ chạy đúng vì 'scripts' nằm ngay trong 'character-llm'
from scripts import scraper

Thư mục cha hiện tại: d:\Small-Chatbot_RAG\character-llm
Danh sách file/folder tại đây: ['.env.example', '.gitignore', 'data', 'inference', 'models', 'notebooks', 'README.md', 'requirements.txt', 'scripts', 'training', '__init__.py']


## 1. Utils Function Test

### Scraper

Scraper sẽ truy cập vào url được quy định, lọc các html của website để chọn ra và trả về raw-text ban đầu về thông tin nhân vật

In [6]:
from scripts import scraper, cleaner
import json

# 1. Thu thập dữ liệu
data_profile = scraper.scrape_character_wiki(
    "https://genshin-impact.fandom.com/wiki/Kamisato_Ayaka/Profile",
    "Kamisato Ayaka"
)
data_voiceover = scraper.scrape_voiceover_wiki(
    "https://genshin-impact.fandom.com/wiki/Kamisato_Ayaka/Voice-Overs",
    "Kamisato Ayaka"
)

data_q1 = scraper.scrape_character_wiki(
    "https://genshin-impact.fandom.com/wiki/Shuumatsuban_Suspicions","Kamisato Ayaka"
)

# 2. Hợp nhất dữ liệu (Giả sử kết quả trả về là Dictionary)
# Nếu trả về List, hãy dùng: all_data = data_profile + data_voiceover
all_docs = [data_profile, data_voiceover, data_q1]

for idx, doc in enumerate(all_docs):
    print(f"=== DOCUMENT {idx+1} DETAILS ===")
    print(f"ID        : {doc.get('doc_id', 'N/A')}")
    print(f"Character : {doc.get('character', 'N/A')}")
    print(f"Total Sect: {len(doc.get('sections', []))}")
    print("-" * 50)
    
    # In chi tiết từng Section
    for i, s in enumerate(doc.get('sections', [])):
        title = s.get('section', 'No Title')
        content = s.get('content', '')
        word_count = len(content.split())
        
        print(f"[{i:02d}] {title:<35} | {word_count:>4} words")
        
        # In thử 100 ký tự đầu của nội dung để kiểm tra chất lượng text
        preview = content[:100].replace('\n', ' ').strip()
        print(f"     Preview: {preview}...")
    print("\n")

# Lưu biến raw để dùng cho các bước sau (cleaner)
raw = all_docs

TypeError: _parse_voiceover_tables() missing 1 required positional argument: 'character_name'

In [ ]:
# Debug cell — xem raw HTML table structure
import requests
from bs4 import BeautifulSoup

api_base = "https://genshin-impact.fandom.com/api.php"
params = {
    "action": "parse",
    "page":   "Kamisato_Ayaka/Voice-Overs",
    "prop":   "text",
    "format": "json",
}
headers = {"User-Agent": "CharacterLLM-Scraper/1.0"}

resp = requests.get(api_base, params=params, headers=headers, timeout=15)
html  = resp.json()["parse"]["text"]["*"]
soup  = BeautifulSoup(html, "html.parser")

# Xem header của từng table
for i, table in enumerate(soup.find_all("table")):
    headers_row = table.find("tr")
    if headers_row:
        cols = [th.get_text(strip=True) for th in headers_row.find_all(["th", "td"])]
        print(f"Table [{i}] headers: {cols}")
    
    # Xem 1 data row mẫu
    rows = table.find_all("tr")
    if len(rows) > 1:
        sample = [td.get_text(strip=True)[:60] for td in rows[1].find_all(["td", "th"])]
        print(f"         sample row: {sample}")
    print()
# Debug cell — xem HTML bên trong 1 row của Table [0]
table = soup.find_all("table")[0]
rows  = table.find_all("tr")

# In raw HTML của 2 rows đầu tiên
for row in rows[:3]:
    cells = row.find_all(["td", "th"])
    print(f"--- row ({len(cells)} cells) ---")
    for i, cell in enumerate(cells):
        print(f"  cell[{i}] classes: {cell.get('class')}")
        print(f"  cell[{i}] text   : {cell.get_text(strip=True)[:100]}")
        # Tìm thẻ con nào có text thực
        for child in cell.find_all(["p", "span", "div"], limit=3):
            t = child.get_text(strip=True)
            if t:
                print(f"    └─ <{child.name}> class={child.get('class')}: {t[:80]}")
    print()
table = soup.find_all("table")[0]
rows  = table.find_all("tr")

for row in rows[:8]:
    cells = row.find_all(["td", "th"])
    classes = row.get("class", [])
    print(f"--- row classes={classes} ({len(cells)} cells) ---")
    for i, cell in enumerate(cells):
        text = cell.get_text(separator=" ", strip=True)
        print(f"  cell[{i}] class={cell.get('class')} | {text[:120]}")
    print()

In [ ]:
# Cell 2 - Xem nội dung từng section trước khi clean
doc_index = 0      # 0: Profile, 1: Voice-Overs
section_index = 0  # Chỉ số section trong document đó

# Truy cập đúng cấu trúc: raw[doc_index] trả về 1 dictionary
s = raw[doc_index]['sections'][section_index]

print(f"=== Document: {raw[doc_index]['character']} ({'Profile' if doc_index==0 else 'Voice-Overs'}) ===")
print(f"=== Section: {s['section']} ===")
print("-" * 30)
print(repr(s['content'][:500]))  # Xem 500 ký tự đầu tiên

In [ ]:
# Cell 3 - Clean từng phần
cleaned_list = []

for doc in raw:
    cleaned_doc = cleaner.clean_document(doc)
    cleaned_list.append(cleaned_doc)
    
    # Phân tích sự thay đổi cho từng doc
    orig_sections = {s['section'] for s in doc['sections']}
    kept_sections = {s['section'] for s in cleaned_doc['sections']}
    dropped = orig_sections - kept_sections
    
    print(f"--- Character: {doc['character']} ---")
    print(f"Trước  : {len(orig_sections)} sections")
    print(f"Sau    : {len(kept_sections)} sections")
    print(f"Dropped: {dropped if dropped else 'None'}\n")

# Lưu lại để dùng tiếp
cleaned = cleaned_list

In [ ]:
# Chọn document muốn xem (0 là Profile, 1 là Voice-Overs)
doc_idx = 0 

# Lấy các section đã clean của doc đó
for s in cleaned[doc_idx]['sections']:
    # Tìm lại nội dung gốc tương ứng trong raw
    original = next(x for x in raw[doc_idx]['sections'] if x['section'] == s['section'])
    
    print(f"\n{'='*60}")
    print(f"SECTION: {s['section']}")
    print(f"--- BEFORE ({len(original['content'])} chars) ---")
    print(repr(original['content'][:300])) 
    
    print(f"--- AFTER  ({len(s['content'])} chars) ---")
    print(repr(s['content'][:300]))

### Save raw-text

### Chunker
Chunker có nhiệm vụ chia raw-text dài thành nhiều đoạn nhỏ (chunks), đồng thời giữ một phần overlap giữa các chunk liên tiếp để tránh mất ngữ cảnh khi embedding, retrieval hoặc training.

Tập chunked sẽ được gói lại bằng JSON với nội dung các thành phần theo dạng danh sách

In [ ]:
# Cell 5 - Chunker (cách tốt hơn)
from scripts import chunker

all_chunks = []
for doc in cleaned:
    chunks = chunker.chunk_document(doc, chunk_size=200, overlap_entries=40)
    all_chunks.extend(chunks)

chunker.save_chunks(chunks=all_chunks, path="../data/processed/chunks.json")

# Kiểm tra
print(f"Tổng chunks: {len(all_chunks)}")
print()
# Xem phân bố chunks theo source
from collections import Counter
sources = Counter(c['doc_id'] for c in all_chunks)
for doc_id, count in sources.items():
    print(f"  {doc_id}: {count} chunks")